# Coder reliability with reliability-lab: binary, multiple choice and numeric scores

<a href="https://colab.research.google.com/github/Sathapana/reliability-lab/blob/main/notebooks/tutorial_coder_reliability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This tutorial measures how far a panel of coders agree, using Krippendorff's alpha as implemented in [reliability-lab](https://github.com/Sathapana/reliability-lab).

We simulate the coders' input ourselves, so we know the truth behind every rating: **10 coders each rate the same 200 units**. The same panel then answers three kinds of question, in order:

1. **Binary** answers: `A` or `B`.
2. **Multiple choice**: `A`, `B` or `C`.
3. **Numeric scores** from 1 to 10.

Alpha is 1 when coders agree perfectly and 0 when they agree no more than chance would predict. It can be negative when they disagree systematically.

> **Reliability is not validity.** Coders can agree consistently and still be wrong in the same way. Alpha tells you whether the coding is reproducible, not whether it is correct.

## Setup

In [ ]:
%pip install -q reliability-lab matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reliability_lab import CoderSpec, RatingFrame, ReliabilityStudy

pd.set_option("display.precision", 3)

## 1. Generating the coders' input

Each unit has one true answer. Each coder has their own accuracy: the chance that they give the true answer. When a coder does not know, they guess:

- for categories (`A`/`B`, `A`/`B`/`C`) the guess is drawn at random, in proportion to how common each category is;
- for numeric scores the guess is a near miss: the true score shifted by a small random amount (`near_miss` is the spread of that shift).

`spread` controls how much coders differ from one another. `coder_accuracy` lets us fix the accuracy of particular coders, for example to plant one careless coder.

The function returns the **wide** table that coders' input usually comes in: one row per unit, one column per coder.

In [ ]:
def make_ratings(
    values,
    n_units=200,
    n_coders=10,
    accuracy=0.8,
    spread=0.1,
    prevalence=None,
    near_miss=None,
    coder_accuracy=None,
    seed=7,
):
    """Simulated ratings: a units x coders table, the true answers and each coder's accuracy."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=object)
    k = len(values)
    p = np.full(k, 1 / k) if prevalence is None else np.asarray(prevalence, dtype=float)

    truth = rng.choice(k, size=n_units, p=p)
    acc = np.clip(accuracy + spread * rng.standard_normal(n_coders), 0.05, 1.0)
    for coder, a in (coder_accuracy or {}).items():
        acc[coder] = a

    knows = rng.random((n_units, n_coders)) < acc
    if near_miss is None:
        guess = rng.choice(k, size=(n_units, n_coders), p=p)
    else:
        shift = np.rint(rng.normal(0, near_miss, size=(n_units, n_coders))).astype(int)
        guess = np.clip(truth[:, None] + shift, 0, k - 1)
    answer = np.where(knows, truth[:, None], guess)

    units = pd.Index([f"u{i + 1:03d}" for i in range(n_units)], name="unit")
    coders = [f"coder_{j + 1:02d}" for j in range(n_coders)]
    wide = pd.DataFrame(values[answer], index=units, columns=coders)
    return wide, pd.Series(values[truth], index=units, name="truth"), pd.Series(acc, index=coders)

reliability-lab works on **long** data: one row per rating, naming the unit, the coder and the value. These two helpers reshape the wide table and build a study. `human_panel` is the design for a panel of coders, and `CoderSpec(by="rater_id")` says that each rater column is one coder.

All our coders rate every unit, so nothing is missing; we ask for one missing-data policy to keep the output short.

In [ ]:
def to_long(wide):
    return wide.reset_index().melt(id_vars="unit", var_name="coder", value_name="answer")


def make_study(wide, measurement, values):
    ratings = RatingFrame.from_long(to_long(wide), unit="unit", rater="coder", value="answer")
    return ReliabilityStudy(
        ratings,
        measurement=measurement,
        values=values,
        coders=CoderSpec(by="rater_id"),
        design="human_panel",
        missing_policies=["as_missing"],
    )


def percent_agreement(wide):
    """Share of coder pairs that give the same answer, averaged over units."""
    m = wide.shape[1]
    same = sum(
        (wide.iloc[:, i] == wide.iloc[:, j]).mean() for i in range(m) for j in range(i + 1, m)
    )
    return same / (m * (m - 1) / 2)

## 2. Binary answers: A or B

Ten coders, 200 units, a mean accuracy of 0.8, and `A` and `B` equally common.

In [ ]:
binary, binary_truth, binary_acc = make_ratings(["A", "B"])
print(binary.shape)
binary.head()

In [ ]:
print("Accuracy of each coder:")
print(binary_acc.round(2).to_string())
print()
print("Answers per coder:")
binary.apply(pd.Series.value_counts).T

In [ ]:
to_long(binary).head()

### Alpha

Binary answers are categories with no order, so the measurement is **nominal**: any two different answers count as a full disagreement.

In [ ]:
binary_study = make_study(binary, "nominal", ["A", "B"])
binary_result = binary_study.alpha()
print(binary_result.summary())

Alpha is `1 - D_o / D_e`:

- `D_o` is the **observed** disagreement: the share of pairs of ratings within a unit that differ.
- `D_e` is the disagreement **expected** by chance: how often two ratings drawn at random from all the ratings would differ.

The **coincidence matrix** counts the pairs of values found within units. Agreements sit on the diagonal.

In [ ]:
estimate = binary_result["as_missing"]
estimate.coincidence

In [ ]:
print(f"D_o = {estimate.d_o:.3f}, D_e = {estimate.d_e:.3f}, alpha = 1 - D_o/D_e = {1 - estimate.d_o / estimate.d_e:.3f}")

Each unit's contribution to `D_o` is listed in `per_unit`, so the units the coders disagree on most are easy to find. The closer a unit's split is to 5 against 5, the higher its disagreement.

In [ ]:
hardest = estimate.per_unit.sort_values("disagreement", ascending=False).head(5)
hardest.join(binary, on="unit_id")

### How precise is alpha?

An alpha from 200 units is an estimate, so it needs an interval. reliability-lab always asks which **scheme** to resample by, because the schemes answer different questions:

- `scheme="units"` resamples the units and holds the coders fixed: *these 10 coders on other units like these*.
- `scheme="two_way"` resamples units and coders: *new coders like these on new units*. It is usually wider.

In [ ]:
for scheme in ("units", "two_way"):
    boot = binary_result.bootstrap(scheme=scheme, n_resamples=1000, random_state=7)
    print(boot.summary())
    print()

### Which coders drive alpha?

The leave-one-coder-out jackknife recomputes alpha without each coder in turn. A positive `influence` means the coder raises alpha; a coder who disagrees with the rest has a negative influence. Compare it with the accuracies we simulated.

In [ ]:
jk = binary_result.coder_jackknife()
influence = jk.influence.set_index("coder")[["alpha_without", "influence"]]
influence.assign(true_accuracy=binary_acc).sort_values("influence").round(3)

### Percent agreement can mislead

When one category is much more common than the other, coders agree often just by guessing the common answer. Percent agreement rewards that; alpha corrects for it. Here the same coders rate a set in which 90% of the units are `A`: percent agreement goes up, alpha goes down.

In [ ]:
skewed, _, _ = make_ratings(["A", "B"], prevalence=[0.9, 0.1])
for name, wide in [("balanced", binary), ("90% A", skewed)]:
    a = make_study(wide, "nominal", ["A", "B"]).alpha().alpha
    print(f"{name:<9} percent agreement {percent_agreement(wide):.3f}   alpha {a:.3f}")

**Try it:** lower `accuracy` to 0.6, or raise `spread` to 0.3 so the coders differ more, and rerun the cells above. Watch how alpha, its intervals and the jackknife influence change.

## 3. Multiple choice: A, B or C

The same panel now chooses between three answers. The answers still have no order, so the measurement is still **nominal**.

To see the jackknife at work, we plant one careless coder: `coder_04` knows the answer only 30% of the time.

In [ ]:
choice, choice_truth, choice_acc = make_ratings(["A", "B", "C"], coder_accuracy={3: 0.3})
choice.head()

In [ ]:
choice_result = make_study(choice, "nominal", ["A", "B", "C"]).alpha()
print(choice_result.summary())

In [ ]:
choice_result["as_missing"].coincidence

With three categories a random guess matches less often, so chance disagreement `D_e` is higher (about 2/3 rather than 1/2).

The jackknife picks out the careless coder: alpha rises when `coder_04` is left out.

In [ ]:
jk = choice_result.coder_jackknife()
influence = jk.influence.set_index("coder")[["alpha_without", "influence"]]
influence.assign(true_accuracy=choice_acc).sort_values("influence").round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
influence["influence"].plot.bar(ax=ax, color=["tab:red" if v < 0 else "tab:blue" for v in influence["influence"]])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("alpha - alpha without coder")
ax.set_title("Coder influence, multiple choice")
plt.tight_layout()
plt.show()

In [ ]:
print(choice_result.bootstrap(scheme="two_way", n_resamples=1000, random_state=7).summary())

**Try it:** set `coder_accuracy={3: 0.3, 7: 0.3}` to add a second careless coder, or make `C` rare with `prevalence=[0.45, 0.45, 0.1]`.

## 4. Numeric scores from 1 to 10

Now each coder gives a score from 1 to 10. Coders know the exact score only half the time, but when they miss they are usually close: a score of 7 might be rated 6 or 8, rarely 2.

In [ ]:
scores = list(range(1, 11))
numeric, numeric_truth, numeric_acc = make_ratings(scores, accuracy=0.5, near_miss=1.5)
numeric = numeric.astype(int)
numeric.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
errors = numeric.sub(numeric_truth.astype(int), axis=0).stack()
errors.value_counts().sort_index().plot.bar(ax=ax)
ax.set_xlabel("rating - true score")
ax.set_ylabel("ratings")
ax.set_title("Most errors are near misses")
plt.tight_layout()
plt.show()

### The measurement level matters

With numbers, the measurement level decides how much a disagreement costs:

- **nominal**: 6 versus 7 counts as much as 1 versus 10;
- **ordinal**: only the order of scores counts, weighted by how many ratings lie between them;
- **interval**: the cost is the squared difference, so 6 versus 7 costs 1 and 1 versus 10 costs 81.

The data are the same in all three rows; only the question changes.

In [ ]:
rows = []
for level in ("nominal", "ordinal", "interval"):
    est = make_study(numeric, level, scores).alpha()["as_missing"]
    rows.append({"measurement": level, "alpha": est.alpha, "D_o": est.d_o, "D_e": est.d_e})
pd.DataFrame(rows).set_index("measurement")

Only the alpha column compares across rows: each level measures distance in its own units, so `D_o` and `D_e` are on different scales.

Nominal alpha is low, because exact matches are rare. Ordinal and interval alpha are high, because the scores are close. For scores like these, choose the level that matches how the scores will be used: `interval` if differences between scores are meaningful, `ordinal` if only their order is.

In [ ]:
numeric_result = make_study(numeric, "interval", scores).alpha()
print(numeric_result.summary())

In [ ]:
print(numeric_result.bootstrap(scheme="two_way", n_resamples=1000, random_state=7).summary())

The coincidence matrix for scores is concentrated around the diagonal: pairs of near misses.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
coin = numeric_result["as_missing"].coincidence
im = ax.imshow(coin.to_numpy(), cmap="Blues")
ax.set_xticks(range(len(scores)), scores)
ax.set_yticks(range(len(scores)), scores)
ax.set_xlabel("score")
ax.set_ylabel("score")
ax.set_title("Coincidence matrix, scores 1-10")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

**Try it:** widen the near misses with `near_miss=3`, and see interval alpha fall while nominal alpha barely moves.

## 5. Summary

The three cases side by side:

In [ ]:
summary = pd.DataFrame(
    [
        {"case": "binary A/B", "measurement": "nominal", "alpha": binary_result.alpha},
        {"case": "multiple choice A/B/C", "measurement": "nominal", "alpha": choice_result.alpha},
        {"case": "scores 1-10", "measurement": "interval", "alpha": numeric_result.alpha},
    ]
).set_index("case")
summary

Results can be saved as strict JSON with their provenance, or as tidy tables:

In [ ]:
report = numeric_result.to_json()
print(report[:500], "...")
numeric_result.tables()["estimates"]

### Where to go next

- Apply this to your own data: load it into the long format (`unit`, `coder`, `answer`) and call `RatingFrame.from_long`.
- Failed or skipped ratings have their own statuses (`abstained`, `parse_error`, ...) and missing-data policies.
- For LLM judges, designs such as `repeatability` and `prompt_robustness` report evidence beside alpha.
- To choose a bootstrap scheme, see the [bootstrap coverage guidance](https://github.com/Sathapana/reliability-lab/blob/main/docs/guidance/bootstrap-coverage.md).

See the [README](https://github.com/Sathapana/reliability-lab#readme) for the full workflow.